# Notebook Train an SVM Digit Classifier & Export with Pickle


This notebook:
1. loads a real handwritten-digit **dataset from the web** (MNIST),
2. reuses the **exact preprocessing** from Notebook Image Processing (`digit_preprocessing.py`),
3. turns images into features (deskewed pixels + **HOG**),
4. trains and tunes a **Support Vector Machine**,
5. evaluates it (accuracy, report, confusion matrix),
6. **pickles a self-contained model bundle** for deployment, and
7. demonstrates **end-to-end inference** on a fresh image using Notebook 1's segmentation.

In [ ]:
# !pip install -q opencv-python-headless scikit-learn scipy matplotlib

import os

if not os.path.exists('digit_preprocessing.py'):
    # Same source as Notebook 1's export — guarantees train/serve parity on Colab.
    _MODULE = r'''
import cv2
import numpy as np
from scipy import ndimage


def to_grayscale(image):
    if isinstance(image, str):
        image = cv2.imread(image, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError('Could not read image from the given path.')
        return image
    if image.ndim == 3:
        return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return image


def binarize(image, method='otsu', blur=True):
    gray = to_grayscale(image)
    if blur:
        gray = cv2.GaussianBlur(gray, (3, 3), 0)
    if method == 'adaptive':
        binary = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, 31, 10)
    else:
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(binary) > 127:
        binary = cv2.bitwise_not(binary)
    return binary


def deskew(img, size=None):
    h, w = img.shape[:2]
    size = size or w
    m = cv2.moments(img)
    if abs(m['mu02']) < 1e-2:
        return img.copy()
    skew = m['mu11'] / m['mu02']
    M = np.float32([[1, skew, -0.5 * size * skew], [0, 1, 0]])
    return cv2.warpAffine(img, M, (size, size),
                          flags=cv2.WARP_INVERSE_MAP | cv2.INTER_LINEAR)


def _shift_to_center_of_mass(img):
    cy, cx = ndimage.center_of_mass(img)
    rows, cols = img.shape
    shiftx = np.round(cols / 2.0 - cx).astype(int)
    shifty = np.round(rows / 2.0 - cy).astype(int)
    M = np.float32([[1, 0, shiftx], [0, 1, shifty]])
    return cv2.warpAffine(img, M, (cols, rows))


def center_and_resize(binary, out_size=28, box=20):
    coords = cv2.findNonZero(binary)
    if coords is None:
        return np.zeros((out_size, out_size), dtype=np.uint8)
    x, y, w, h = cv2.boundingRect(coords)
    digit = binary[y:y + h, x:x + w]
    if w > h:
        new_w, new_h = box, max(1, int(round(h * box / w)))
    else:
        new_h, new_w = box, max(1, int(round(w * box / h)))
    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((out_size, out_size), dtype=np.uint8)
    x0 = (out_size - new_w) // 2
    y0 = (out_size - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = digit
    return _shift_to_center_of_mass(canvas)


def preprocess_digit(image, do_deskew=True, out_size=28, flatten=False, normalize=False):
    binary = binarize(image)
    if do_deskew:
        binary = deskew(binary)
    tile = center_and_resize(binary, out_size=out_size)
    if normalize:
        tile = tile.astype(np.float32) / 255.0
    if flatten:
        return tile.reshape(-1)
    return tile


def segment_digits(image, min_area=80, pad_ratio=0.25, merge_kernel=1,
                   max_aspect=1.1, split_wide=True):
    """Locate individual digits and return (tiles, boxes) sorted left-to-right.

    Key fixes for "two digits captured as one":
      - merge_kernel defaults to 1 (NO dilation) so close digits stay separate.
        Dilation was bridging neighbouring strokes into a single blob.
      - Any contour wider than `max_aspect * height` is assumed to contain
        multiple touching digits and is split evenly into round(w / digit_w)
        columns (a digit is roughly as wide as it is tall).
    """
    binary = binarize(image)

    # Only dilate if explicitly asked (helps re-join a single BROKEN digit, but
    # also risks merging neighbours — off by default).
    if merge_kernel and merge_kernel > 1:
        kernel = np.ones((merge_kernel, merge_kernel), np.uint8)
        search = cv2.dilate(binary, kernel, iterations=1)
    else:
        search = binary

    contours, _ = cv2.findContours(search, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < min_area or h < 10:
            continue

        # A single digit is ~0.6*h wide. If a box is much wider than tall, it
        # most likely holds several touching digits -> split it into columns.
        if split_wide and w > max_aspect * h:
            n = max(1, int(round(w / (0.6 * h))))   # estimated digit count
            step = w / n
            for k in range(n):
                boxes.append((int(x + k * step), y, int(np.ceil(step)), h))
        else:
            boxes.append((x, y, w, h))

    # Read order = left to right (add y as a tiebreak for multi-row pages).
    boxes.sort(key=lambda b: (b[0]))

    tiles = []
    for (x, y, w, h) in boxes:
        pad = int(pad_ratio * max(w, h))
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(binary.shape[1], x + w + pad), min(binary.shape[0], y + h + pad)
        crop = binary[y0:y1, x0:x1]
        tiles.append(preprocess_digit(crop, do_deskew=True))
    return tiles, boxesdef segment_digits(image, min_area=80, pad_ratio=0.25, merge_kernel=1,
                   max_aspect=1.1, split_wide=True):
    """Locate individual digits and return (tiles, boxes) sorted left-to-right.

    Key fixes for "two digits captured as one":
      - merge_kernel defaults to 1 (NO dilation) so close digits stay separate.
        Dilation was bridging neighbouring strokes into a single blob.
      - Any contour wider than `max_aspect * height` is assumed to contain
        multiple touching digits and is split evenly into round(w / digit_w)
        columns (a digit is roughly as wide as it is tall).
    """
    binary = binarize(image)

    # Only dilate if explicitly asked (helps re-join a single BROKEN digit, but
    # also risks merging neighbours — off by default).
    if merge_kernel and merge_kernel > 1:
        kernel = np.ones((merge_kernel, merge_kernel), np.uint8)
        search = cv2.dilate(binary, kernel, iterations=1)
    else:
        search = binary

    contours, _ = cv2.findContours(search, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w * h < min_area or h < 10:
            continue

        # A single digit is ~0.6*h wide. If a box is much wider than tall, it
        # most likely holds several touching digits -> split it into columns.
        if split_wide and w > max_aspect * h:
            n = max(1, int(round(w / (0.6 * h))))   # estimated digit count
            step = w / n
            for k in range(n):
                boxes.append((int(x + k * step), y, int(np.ceil(step)), h))
        else:
            boxes.append((x, y, w, h))

    # Read order = left to right (add y as a tiebreak for multi-row pages).
    boxes.sort(key=lambda b: (b[0]))

    tiles = []
    for (x, y, w, h) in boxes:
        pad = int(pad_ratio * max(w, h))
        x0, y0 = max(0, x - pad), max(0, y - pad)
        x1, y1 = min(binary.shape[1], x + w + pad), min(binary.shape[0], y + h + pad)
        crop = binary[y0:y1, x0:x1]
        tiles.append(preprocess_digit(crop, do_deskew=True))
    return tiles, boxes
'''
    with open('digit_preprocessing.py', 'w') as f:
        f.write(_MODULE)
    print('Regenerated digit_preprocessing.py for this Colab runtime.')
else:
    print('Found existing digit_preprocessing.py (from Notebook 1).')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time, pickle, platform

# Reuse the SINGLE source of preprocessing truth from Notebook 1.
# (Make sure Notebook 1 has been run so digit_preprocessing.py exists in this folder.)
import digit_preprocessing as dp

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Imports OK.')

In [ ]:
# Fetch MNIST. as_frame=False returns NumPy arrays directly (faster, less memory).
print('Downloading / loading MNIST (first run may take a minute)...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data.astype(np.uint8)        # shape (70000, 784), pixel values 0..255
y = mnist.target.astype(np.int64)      # shape (70000,), labels 0..9
print('Images:', X.shape, '| Labels:', y.shape, '| Classes:', np.unique(y))

# Peek at a few samples.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, img, lab in zip(axes.ravel(), X[:16], y[:16]):
    ax.imshow(img.reshape(28, 28), cmap='gray')
    ax.set_title(str(lab)); ax.axis('off')
plt.suptitle('Raw MNIST samples'); plt.tight_layout(); plt.show()

In [ ]:
def deskew_dataset(flat_images):
    """Apply Notebook 1's deskew to a stack of flattened 28x28 images."""
    out = np.empty_like(flat_images)
    for i, flat in enumerate(flat_images):
        img = flat.reshape(28, 28)
        out[i] = dp.deskew(img).reshape(-1)
    return out

t0 = time.time()
X_desk = deskew_dataset(X)
print(f'Deskewed {len(X_desk):,} images in {time.time() - t0:.1f}s')

# Visual sanity check: top row raw, bottom row deskewed.
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for k in range(8):
    axes[0, k].imshow(X[k].reshape(28, 28), cmap='gray'); axes[0, k].axis('off')
    axes[1, k].imshow(X_desk[k].reshape(28, 28), cmap='gray'); axes[1, k].axis('off')
axes[0, 0].set_ylabel('raw'); axes[1, 0].set_ylabel('deskewed')
plt.suptitle('Raw (top) vs deskewed (bottom)'); plt.tight_layout(); plt.show()

In [ ]:
import cv2

# HOG configured for 28x28 tiles: 14x14 cells, 9 orientation bins.
_hog = cv2.HOGDescriptor(_winSize=(28, 28), _blockSize=(14, 14),
                         _blockStride=(7, 7), _cellSize=(14, 14), _nbins=9)

def hog_features(flat_images):
    """Map flattened 28x28 uint8 images -> HOG feature vectors."""
    feats = []
    for flat in flat_images:
        img = flat.reshape(28, 28).astype(np.uint8)
        feats.append(_hog.compute(img).ravel())
    return np.asarray(feats, dtype=np.float32)

t0 = time.time()
X_hog = hog_features(X_desk)
print(f'HOG features: {X_hog.shape} computed in {time.time() - t0:.1f}s')

In [ ]:
# Standard MNIST split: first 60k train, last 10k test.
X_tr_full, X_te = X_hog[:60000], X_hog[60000:]
y_tr_full, y_te = y[:60000], y[60000:]

TRAIN_SIZE = 12000     # <- raise (or set None) for higher accuracy at the cost of time
if TRAIN_SIZE is not None and TRAIN_SIZE < len(X_tr_full):
    # Stratified subsample keeps class balance.
    X_tr, _, y_tr, _ = train_test_split(
        X_tr_full, y_tr_full, train_size=TRAIN_SIZE,
        stratify=y_tr_full, random_state=RANDOM_STATE)
else:
    X_tr, y_tr = X_tr_full, y_tr_full

print(f'Train: {X_tr.shape} | Test: {X_te.shape}')

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),                 # zero-mean/unit-var per feature
    ('svm', SVC(kernel='rbf', cache_size=1000)),  # RBF handles non-linear digit shapes
])

param_grid = {
    'svm__C': [5, 10],
    'svm__gamma': ['scale', 0.05],
}

grid = GridSearchCV(pipe, param_grid, cv=3, n_jobs=-1, verbose=1, scoring='accuracy')
t0 = time.time()
grid.fit(X_tr, y_tr)
print(f'\nGrid search done in {time.time() - t0:.1f}s')
print('Best params    :', grid.best_params_)
print('Best CV accuracy:', round(grid.best_score_, 4))
model = grid.best_estimator_       # the fitted scaler+SVM pipeline

In [ ]:
t0 = time.time()
y_pred = model.predict(X_te)
print(f'Predicted {len(X_te):,} test images in {time.time() - t0:.1f}s')

acc = accuracy_score(y_te, y_pred)
print(f'\nTest accuracy: {acc:.4f}\n')
print(classification_report(y_te, y_pred, digits=4))

cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion matrix (acc={acc:.3f})'); plt.tight_layout(); plt.show()

In [ ]:
# Inspect a few mistakes — these reveal the genuinely ambiguous digits (4/9, 3/5, 7/1).
wrong = np.where(y_pred != y_te)[0][:8]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2.6))
    for ax, idx in zip(np.atleast_1d(axes), wrong):
        ax.imshow(X[60000 + idx].reshape(28, 28), cmap='gray')
        ax.set_title(f'true {y_te[idx]}\npred {y_pred[idx]}', fontsize=9); ax.axis('off')
    plt.suptitle('Sample misclassifications'); plt.tight_layout(); plt.show()
else:
    print('No misclassifications in the sample slice.')

In [ ]:
bundle = {
    'model': model,                       # sklearn Pipeline(StandardScaler + SVC)
    'feature': 'hog',                     # how raw tiles become features
    'hog_params': dict(winSize=(28, 28), blockSize=(14, 14),
                       blockStride=(7, 7), cellSize=(14, 14), nbins=9),
    'input_shape': (28, 28),
    'classes': list(range(10)),
    'preprocessing_module': 'digit_preprocessing.py',
    'test_accuracy': float(acc),
    'sklearn_version': __import__('sklearn').__version__,
    'python_version': platform.python_version(),
}

with open('svm_digit_model.pkl', 'wb') as f:
    pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
print('Saved svm_digit_model.pkl  (test acc =', round(acc, 4), ')')

In [ ]:
# Reload strictly from disk to simulate a separate deployment process.
with open('svm_digit_model.pkl', 'rb') as f:
    loaded = pickle.load(f)
clf = loaded['model']
hp = loaded['hog_params']
hog = cv2.HOGDescriptor(_winSize=hp['winSize'], _blockSize=hp['blockSize'],
                        _blockStride=hp['blockStride'], _cellSize=hp['cellSize'],
                        _nbins=hp['nbins'])

def predict_image(image):
    """Full inference: segment -> HOG -> SVM. Returns the recognised digit string + boxes."""
    tiles, boxes = dp.segment_digits(image)
    if not tiles:
        return '', []
    feats = np.asarray([hog.compute(t.astype(np.uint8)).ravel() for t in tiles],
                       dtype=np.float32)
    preds = clf.predict(feats)
    return ''.join(map(str, preds)), boxes

# Build a fresh strip (same renderer as Notebook 1) and recognise it.
def make_demo_image(text, size=(160, 700)):
    img = np.zeros(size, dtype=np.uint8); x = 40
    rng = np.random.default_rng(7)
    for ch in text:
        scale = rng.uniform(3.5, 5.0); thick = int(rng.integers(6, 10)); yy = int(rng.integers(95, 120))
        cv2.putText(img, ch, (x, yy), cv2.FONT_HERSHEY_SIMPLEX, scale, 255, thick, cv2.LINE_AA)
        x += int(70 * scale / 4)
    return img

truth = '2026'
test_img = make_demo_image(truth)
pred_str, boxes = predict_image(test_img)
print(f'Ground truth : {truth}')
print(f'Prediction   : {pred_str}')

vis = cv2.cvtColor(test_img, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in boxes:
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)
plt.figure(figsize=(10, 3)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f'Predicted: {pred_str}'); plt.axis('off'); plt.tight_layout(); plt.show()